# Validation Notebook


Data-dependent sanity checks for the feature-engineering pipeline. Run this after `scripts/build_features.py` has produced the preprocessed dataframes under `data/processed/`. If every cell here completes without an `AssertionError`, the pipeline is healthy.

Lower-level unit tests on hand-crafted scenarios live in the modules themselves (e.g. `python -m src.hand_evaluator`).

## Setup

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.cards import HAND_RANKS, CATEGORY_NAMES
from src.hand_evaluator import best_hand_strength
from src.max_strength import max_future_strength_fast
from src.nut_strength import nut_strength

PROCESSED = 'data/processed'

## Load preprocessed dataframes

In [ ]:
# We need the rank columns from each stage and the raw card columns
# (re-derived from the simplified frame) for spot-check recomputation.
df_simplified = pd.read_parquet(f'{PROCESSED}/df_simplified.parquet')
df_nut = pd.read_parquet(f'{PROCESSED}/df_nut_strength_final.parquet')

# Attach the raw card / street columns so we can recompute on samples
df_nut = df_nut.join(df_simplified[
    ['holding', 'board_flop', 'board_turn', 'board_river', 'evaluation_at']
])
df_nut = df_nut.rename(columns={'evaluation_at': 'street_name'})
print(f'Loaded {len(df_nut):,} rows')
df_nut.head()

## Validation 1 — `max_future` invariants

- Max future rank should never be *worse* than current rank (can't draw to a worse hand).
- On river rows, max future rank must equal current rank (no cards left to come).
- Output columns shouldn't have nulls.
- On flop rows, a clear majority should show strict improvement.

In [ ]:
# Test 1: max_future_rank >= hand_rank for every row
violations = df_nut[df_nut['max_future_rank'] < df_nut['hand_rank']]
assert len(violations) == 0, \
    f'{len(violations)} rows where max_future_rank < hand_rank:\n' \
    f'{violations.head()}'
print(f'✓ max_future_rank >= hand_rank for all {len(df_nut):,} rows')

In [ ]:
# Test 2: on river rows, max_future_rank == hand_rank
river_rows = df_nut[df_nut['street_name'] == 'River']
river_mismatches = river_rows[
    river_rows['hand_rank'] != river_rows['max_future_rank']
]
assert len(river_mismatches) == 0, \
    f'{len(river_mismatches)} river rows where ranks differ:\n' \
    f'{river_mismatches.head()}'
print(f'✓ max_future_rank == hand_rank on all {len(river_rows):,} river rows')

In [ ]:
# Test 3: no nulls in the rank columns
for col in ['hand_rank', 'max_future_rank', 'opponent_nut_rank']:
    n_null = df_nut[col].isna().sum()
    assert n_null == 0, f'{col} has {n_null} nulls'
print('✓ No nulls in any rank column')

In [ ]:
# Test 4: most flop rows should improve (with 2 cards to come)
flop_rows = df_nut[df_nut['street_name'] == 'Flop']
if len(flop_rows) > 0:
    improved = flop_rows[flop_rows['max_future_rank'] > flop_rows['hand_rank']]
    pct_improved = 100 * len(improved) / len(flop_rows)
    print(f'ℹ {pct_improved:.1f}% of flop rows show strict improvement '
          f'({len(improved):,}/{len(flop_rows):,})')
    assert pct_improved > 50, 'Suspiciously few flop rows improved'
    print(f'✓ Majority of flop rows improve as expected')

## Validation 2 — `nut_strength` blocker effects

- We expect some rows where the player's max-future > opponent's nut (blocker effect — player holds key cards no opponent can have).
- These should concentrate on the river, where future cards can't bail opponents out.

In [ ]:
# Identify blocker-effect rows
blocker_rows = df_nut[df_nut['max_future_rank'] > df_nut['opponent_nut_rank']]
print(f'ℹ {len(blocker_rows):,} rows show blocker effect '
      f"(max_future > nut)")
if len(blocker_rows) > 0:
    print('\nBlocker cases by street:')
    print(blocker_rows['street_name'].value_counts())

## Validation 3 — Spot-check recomputation

- Re-run the strength functions on a small random sample
- Confirm the original-row hand category lines up with what the
  evaluator returns now (a sanity check that nothing has drifted)

In [ ]:
sample = df_nut.sample(n=200, random_state=42).copy()

def recompute_category(row):
    t = best_hand_strength(
        row['holding'], row['board_flop'],
        row['board_turn'], row['board_river'],
        row['street_name'],
    )
    return CATEGORY_NAMES[t[0]] if t else None

sample['recomputed_category'] = sample.apply(recompute_category, axis=1)
mismatches = sample[sample['recomputed_category'] != sample['hand_category']]
assert len(mismatches) == 0, \
    f'{len(mismatches)} mismatches between stored hand_category and ' \
    f'recomputed:\n{mismatches[["holding", "board_flop", "hand_category", "recomputed_category"]].head()}'
print(f'✓ All {len(sample)} sampled hand_category values match recomputation')

## All validations passed

If every cell above ran without an `AssertionError`, the strength modules are behaving correctly on real data.